| **Nonlinear Model** | **Linearized MILP Model** |
| :--- | :--- |
| **Decision Variables:**<br>$x_{i,k} \in \{0, 1\}$ | **Decision Variables:**<br>$x_{i,k} \in \{0, 1\}$<br>$y_{i,j,k} \in \{0, 1\}$ |
| **Objective Function:**<br>$$\max \sum_{i < j} S_{i,j} \left( \sum_{k \in T} x_{i,k} \cdot x_{j,k} \right)$$ | **Objective Function:**<br>$$\max \sum_{i < j} \sum_{k \in T} S_{i,j} \cdot y_{i,j,k}$$ |
| **Base Constraints:**<br>1. $\sum_{k \in T} x_{i,k} = 1 \quad \forall i \in G$<br>2. $\sum_{i \in G} x_{i,k} \le C_k \quad \forall k \in T$ | **Base Constraints:**<br>1. $\sum_{k \in T} x_{i,k} = 1 \quad \forall i \in G$<br>2. $\sum_{i \in G} x_{i,k} \le C_k \quad \forall k \in T$ |
| **Enemy Constraint:**<br>$$x_{i,k} \cdot x_{j,k} = 0 \quad \forall k \in T, (i,j) \in E$$ | **Enemy Constraint:**<br>$$x_{i,k} + x_{j,k} \le 1 \quad \forall k \in T, (i,j) \in E$$ |
| **Linearization Constraints:**<br>None | **Linearization Constraints ($y_{i,j,k} = x_{i,k} \cdot x_{j,k}$):**<br>$$\begin{aligned} y_{i,j,k} &\le x_{i,k} \\ y_{i,j,k} &\le x_{j,k} \\ y_{i,j,k} &\ge x_{i,k} + x_{j,k} - 1 \end{aligned}$$ |

In [2]:
from ortools.sat.python import cp_model 

In [3]:
guest_count = 4
table_count = 2

affinity_scores = [
    [0, 10, 2, 0], 
    [10, 0, 0, 1], 
     [2, 0, 0, 8], 
     [0, 1, 8, 0]]

table_capacities = [2, 2]
enemies = [(0, 2)]

In [4]:
model = cp_model.CpModel()

In [5]:
x = {}
for i in range(guest_count):
    for k in range(table_count):
        x[i,k] = model.NewBoolVar(f"x[{i}][{k}]")

In [6]:
for i in range(guest_count):
    model.AddExactlyOne([x[i,k] for k in range(table_count)])

In [7]:
for k in range(table_count):
    this_line = []
    for i in range(guest_count):
        this_line.append(x[i,k])
    model.Add(sum(this_line) <= table_capacities[k])

#list comp
# for k in range(table_count):
#     capacity_k = table_capacities[k]
#     model.Add(sum(x[i,k] for i in range(guest_count)) <= capacity_k)

In [8]:
for k in range(table_count):
    for i,j in enemies:
        model.Add(x[i,k]+x[j,k] <= 1)

In [9]:
obj = []
for i in range(guest_count):
    for j in range(i+1, guest_count):
        score = affinity_scores[i][j]
        if score > 0:
            for k in range(table_count):
                same_table = model.NewBoolVar("same_table[{i}][{j}][{k}]")
                model.AddBoolAnd([x[i,k], x[j,k]]).OnlyEnforceIf(same_table)
                obj.append(score*same_table)

In [10]:
model.Maximize(sum(obj))

In [17]:
solver = cp_model.CpSolver()
status = solver.Solve(model)

print(status)
print(solver.ObjectiveValue())
for k in range(table_count):
    seated = [ i for i in range(guest_count) if solver.Value(x[i, k]) == 1 ]
    print( f"Table {k + 1} (Capacity {table_capacities[k]}): Guests {seated}" )

print(solver.ResponseStats())

CpSolverStatus.OPTIMAL
18.0
Table 1 (Capacity 2): Guests [0, 1]
Table 2 (Capacity 2): Guests [2, 3]
CpSolverResponse summary:
status: OPTIMAL
objective: 18
best_bound: 18
integers: 5
booleans: 4
conflicts: 0
branches: 10
propagations: 10
integer_propagations: 22
restarts: 0
lp_iterations: 0
walltime: 0.00650624
usertime: 0.00650634
deterministic_time: 2.8122e-05
gap_integral: 1.96506e-05
solution_fingerprint: 0xe1a28285024c9a40

